In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(".")), "Code"))
from pathlib import Path
import pandas as pd


# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'LIS' / 'code'
mail_root = Path('/Users/jedrek/Library/Mail/V10')

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))


## Scan Mail for LIS result emails

Walks the entire macOS Mail store (`~/Library/Mail/V10`) and collects all `.emlx` messages from `postbox@lisdatacenter.org`, then displays them sorted chronologically.

In [2]:
import email
import email.policy
import re
import html
from html.parser import HTMLParser
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone, timedelta

LIS_SENDER = "postbox@lisdatacenter.org"

# Only collect emails after this moment (inclusive): 2026-03-07 00:00 UTC+1
CUTOFF = datetime(2026, 3, 7, 0, 0, 0, tzinfo=timezone(timedelta(hours=1)))

def parse_emlx(path: Path):
    """Parse a macOS .emlx file (first line = byte count, then RFC 2822 email)."""
    try:
        raw = path.read_bytes()
        newline_pos = raw.index(b"\n")
        byte_count = int(raw[:newline_pos].strip())
        email_bytes = raw[newline_pos + 1 : newline_pos + 1 + byte_count]
        return email.message_from_bytes(email_bytes, policy=email.policy.default)
    except Exception:
        return None


def _decode_payload(part) -> str:
    """Decode a message part to a string, handling base64/QP encodings."""
    payload = part.get_payload(decode=True)
    if not payload:
        return ""
    charset = part.get_content_charset() or "utf-8"
    return payload.decode(charset, errors="replace")


class _HTMLToText(HTMLParser):
    """Convert HTML to readable plain text, similar to what a mail app shows."""

    BLOCK_TAGS = {"p", "div", "br", "li", "tr", "h1", "h2", "h3", "h4", "h5", "h6",
                  "blockquote", "pre", "hr", "table", "thead", "tbody", "tfoot"}
    SKIP_TAGS  = {"script", "style", "head"}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self._parts = []
        self._skip  = 0

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        if tag in self.SKIP_TAGS:
            self._skip += 1
        elif tag in self.BLOCK_TAGS:
            self._parts.append("\n")

    def handle_endtag(self, tag):
        tag = tag.lower()
        if tag in self.SKIP_TAGS:
            self._skip = max(0, self._skip - 1)
        elif tag in self.BLOCK_TAGS:
            self._parts.append("\n")

    def handle_data(self, data):
        if self._skip == 0:
            self._parts.append(data)

    def get_text(self) -> str:
        text = "".join(self._parts)
        lines = [l.rstrip() for l in text.splitlines()]
        cleaned = []
        blank_run = 0
        for line in lines:
            if line == "":
                blank_run += 1
                if blank_run <= 1:
                    cleaned.append("")
            else:
                blank_run = 0
                cleaned.append(line)
        return "\n".join(cleaned).strip()


def _html_to_text(html_str: str) -> str:
    parser = _HTMLToText()
    parser.feed(html_str)
    return parser.get_text()


def get_body(msg) -> str:
    """Extract the best plain-text body. Prefers text/plain; falls back to text/html."""
    plain_parts = []
    html_parts  = []

    if msg.is_multipart():
        for part in msg.walk():
            if part.is_multipart():
                continue
            if "attachment" in str(part.get("Content-Disposition", "")):
                continue
            ct = part.get_content_type()
            if ct == "text/plain":
                plain_parts.append(_decode_payload(part))
            elif ct == "text/html":
                html_parts.append(_decode_payload(part))
    else:
        ct = msg.get_content_type()
        if ct == "text/plain":
            plain_parts.append(_decode_payload(msg))
        elif ct == "text/html":
            html_parts.append(_decode_payload(msg))

    if plain_parts:
        return "\n".join(plain_parts).strip()
    if html_parts:
        return _html_to_text("\n".join(html_parts))

    raw = msg.get_payload(decode=True)
    if raw:
        return raw.decode("utf-8", errors="replace").strip()
    return ""


records = []

for emlx_path in mail_root.rglob("*.emlx"):
    if emlx_path.name.endswith(".partial.emlx"):
        continue
    msg = parse_emlx(emlx_path)
    if msg is None:
        continue

    sender = str(msg.get("From", ""))
    if LIS_SENDER not in sender.lower():
        continue

    date_str = str(msg.get("Date", ""))
    try:
        dt = parsedate_to_datetime(date_str)
        # Normalise timezone-naive datetimes to UTC for comparison
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
    except Exception:
        dt = None

    # Skip emails before 2026-03-07 00:00 UTC+1
    if dt is None or dt < CUTOFF:
        continue

    records.append({
        "date":    dt,
        "subject": str(msg.get("Subject", "(no subject)")),
        "body":    get_body(msg),
        "file":    str(emlx_path),
    })

print(f"Found {len(records)} email(s) from {LIS_SENDER} after {CUTOFF.strftime('%Y-%m-%d %H:%M %Z')}")


Found 94 email(s) from postbox@lisdatacenter.org after 2026-03-07 00:00 UTC+01:00


In [3]:
df = pd.DataFrame(records)

if df.empty:
    print("No emails found.")
else:
    # Sort chronologically (NaT dates go last)
    df = df.sort_values("date", na_position="last").reset_index(drop=True)
    df.index += 1  # 1-based numbering

    # Display summary table
    display(df[["date", "subject"]].rename(columns={"date": "Date", "subject": "Subject"}))


,Date,Subject
1,2026-03-07 12:25:07+01:00,job 1453214 region_c test
2,2026-03-07 12:27:40+01:00,job 1453215 region_c test
3,2026-03-07 12:28:12+01:00,job 1453216 region_c test
4,2026-03-07 12:29:13+01:00,job 1453217 region_c test
5,2026-03-07 12:30:05+01:00,job 1453218 region_c test
...,...,...
90,2026-03-08 06:09:17+01:00,job 1453455 LA group 2019
91,2026-03-08 06:14:42+01:00,job 1453456 LA group 2020
92,2026-03-08 06:20:10+01:00,job 1453457 LA group 2021
93,2026-03-08 06:25:38+01:00,job 1453458 LA group 2022


### Inspect a specific email body

Change `EMAIL_INDEX` (1-based) to read the full body of any listed email.

In [4]:
df.sort_values("date", ascending=False, inplace=True)

In [5]:
EMAIL_INDEX = 1  # change to inspect a different email (1-based)

if not df.empty and EMAIL_INDEX <= len(df):
    row = df.iloc[EMAIL_INDEX - 1]
    print(f"Date   : {row['date']}")
    print(f"Subject: {row['subject']}")
    print(f"File   : {row['file']}")
    print("-" * 60)
    print(row["body"])
else:
    print("No email at that index.")

Date   : 2026-03-08 06:31:10+01:00
Subject: job 1453459 LA group 2023
File   : /Users/jedrek/Library/Mail/V10/67AF7211-575D-461C-848B-04F9ED568CC1/INBOX.mbox/4E96815A-30EB-42EF-B40B-C98AD2E7740C/Data/9/2/Messages/29863.emlx
------------------------------------------------------------
############################### NOTICE TO USERS ###############################
                                                                        Use of the data in the LUXEMBOURG INCOME STUDY DATABASES is governed by regulations which do not allow copying or further distribution of the survey microdata.

Anyone violating these regulations will lose all privileges to the databases and may be subject to prosecution under the law. In addition, any attempt to circumvent the LIS processing system or unauthorized entry into the LIS computing system will result in prosecution.
All papers written using the LUXEMBOURG INCOME STUDY DATABASES must be  submitted for entry into the Working Papers Series. Users o

---
## Parse LA voiv emails → regional DataFrame

Scans all emails with **"LA"** in the subject, extracts every `data_region_YEAR_REGION = { … }` Python-dict block from the R output, and assembles a tidy DataFrame.  One row = one (year, region) combination.

In [6]:
import ast
import re
from datetime import datetime, timezone, timedelta
from html.parser import HTMLParser

# ── Years of interest ────────────────────────────────────────────────────────
LA_YEARS = {1999} | set(range(2004, 2024))   # 1999 + 2004..2023

# ── Regex: matches   data_region_1999_dolnoslaskie = {   ...   }
# The dict body spans multiple lines; we use a non-greedy scan on the
# flattened text.  The outer group captures everything between { and the
# matching } that ends with "}\n" at column 0 (i.e. unindented closing brace).
_BLOCK_RE = re.compile(
    r"data_region_(\d{4})_(\w+)\s*=\s*\{(.+?)\n\}",
    re.DOTALL,
)

def _html_to_text_full(html_str: str) -> str:
    """Reuse the existing _HTMLToText class defined in cell 3."""
    p = _HTMLToText()
    p.feed(html_str)
    return p.get_text()


def _parse_value(raw: str):
    """Convert a single-element list literal like [1999] or ["N/A"] to a scalar."""
    raw = raw.strip()
    try:
        val = ast.literal_eval(raw)
        if isinstance(val, list) and len(val) == 1:
            v = val[0]
            return None if (isinstance(v, str) and v in ("N/A", "NA")) else v
        return val
    except Exception:
        return None


def _parse_dict_body(body: str) -> dict:
    """Parse the key-value lines inside a data_region_... = { … } block."""
    out = {}
    # Each line looks like:   '   \'key\': [value],'
    for m in re.finditer(r"'([^']+)'\s*:\s*(\[[^\]]*\])", body):
        key, raw_val = m.group(1), m.group(2)
        out[key] = _parse_value(raw_val)
    return out


def extract_region_dicts(text: str) -> list[dict]:
    """Find all data_region_YEAR_REGION blocks and return list of flat dicts."""
    rows = []
    for m in _BLOCK_RE.finditer(text):
        year_str, region, body = m.group(1), m.group(2), m.group(3)
        row = _parse_dict_body(body)
        # Ensure year and region are always present (even if not in the dict body)
        row["year"]   = int(year_str)
        row["region"] = region
        rows.append(row)
    return rows


# ── Scan emails ──────────────────────────────────────────────────────────────
all_rows = []
la_mail_index = []   # track which emails were used

for emlx_path in mail_root.rglob("*.emlx"):
    if emlx_path.name.endswith(".partial.emlx"):
        continue
    raw_bytes = emlx_path.read_bytes()
    if b"postbox@lisdatacenter.org" not in raw_bytes:
        continue

    msg = parse_emlx(emlx_path)
    if msg is None:
        continue

    subject = str(msg.get("Subject", ""))
    # Must have "LA" in the subject
    if "LA" not in subject:
        continue

    # Extract the year from the subject
    year_match = re.search(r"\b(1999|200[4-9]|20[12][0-9])\b", subject)
    if year_match is None:
        continue
    year_in_subj = int(year_match.group(1))
    if year_in_subj not in LA_YEARS:
        continue

    body = get_body(msg)          # uses the function from cell 3
    rows = extract_region_dicts(body)
    if not rows:
        continue

    date_str = str(msg.get("Date", ""))
    try:
        mail_dt = parsedate_to_datetime(date_str)
        if mail_dt.tzinfo is None:
            mail_dt = mail_dt.replace(tzinfo=timezone.utc)
    except Exception:
        mail_dt = None

    for row in rows:
        row["_mail_subject"] = subject
        row["_mail_date"]    = mail_dt

    all_rows.extend(rows)
    la_mail_index.append({
        "subject": subject,
        "date":    mail_dt,
        "n_rows":  len(rows),
        "file":    str(emlx_path),
    })

print(f"Parsed {len(all_rows)} region-year rows from {len(la_mail_index)} emails.")
print("\nEmails used:")
for e in sorted(la_mail_index, key=lambda x: (x["date"] or datetime.min.replace(tzinfo=timezone.utc))):
    print(f"  [{e['date']}]  {e['subject']:<40}  → {e['n_rows']} rows")


Parsed 352 region-year rows from 22 emails.

Emails used:
  [2026-03-08 04:25:38+01:00]  job 1453412 LA voiv 1999                  → 16 rows
  [2026-03-08 04:26:58+01:00]  job 1453413 LA voiv 1999                  → 16 rows
  [2026-03-08 04:27:32+01:00]  job 1453414 LA voiv 2004                  → 16 rows
  [2026-03-08 04:28:01+01:00]  job 1453415 LA voiv 2005                  → 16 rows
  [2026-03-08 04:28:30+01:00]  job 1453416 LA voiv 2006                  → 16 rows
  [2026-03-08 04:28:59+01:00]  job 1453417 LA voiv 2007                  → 16 rows
  [2026-03-08 04:29:31+01:00]  job 1453418 LA voiv 2008                  → 16 rows
  [2026-03-08 04:33:35+01:00]  job 1453420 LA voiv 2009                  → 16 rows
  [2026-03-08 04:34:09+01:00]  job 1453422 LA voiv 2010                  → 16 rows
  [2026-03-08 04:34:39+01:00]  job 1453423 LA voiv 2011                  → 16 rows
  [2026-03-08 04:35:07+01:00]  job 1453424 LA voiv 2012                  → 16 rows
  [2026-03-08 04:35:36+01:00]

In [7]:
# ── Deduplicate: if multiple emails cover the same (year, region),
#    keep the one from the most recent email (latest _mail_date).
df_raw = pd.DataFrame(all_rows)

if df_raw.empty:
    print("No data extracted.")
else:
    # Sort so the most recent email comes last → drop_duplicates keeps last
    df_raw = df_raw.sort_values("_mail_date", na_position="first")
    df_la = (
        df_raw
        .drop_duplicates(subset=["year", "region"], keep="last")
        .drop(columns=["_mail_subject", "_mail_date"])
        .sort_values(["year", "region"])
        .reset_index(drop=True)
    )

    # Coerce numeric-looking columns to float (they may be mixed str/float due to "N/A")
    skip_cols = {"year", "region"}
    for col in df_la.columns:
        if col in skip_cols:
            continue
        df_la[col] = pd.to_numeric(df_la[col], errors="coerce")

    print(f"Final DataFrame: {df_la.shape[0]} rows × {df_la.shape[1]} columns")
    print(f"Years covered : {sorted(df_la['year'].unique())}")
    print(f"Regions       : {sorted(df_la['region'].unique())}")
    display(df_la)

df_la.to_csv(data_root.parent / "LIS_Voiv_2000.csv", index=False)

Final DataFrame: 336 rows × 200 columns
Years covered : [np.int64(1999), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Regions       : ['dolnoslaskie', 'kujawsko_pomorskie', 'lodzkie', 'lubelskie', 'lubuskie', 'malopolskie', 'mazowieckie', 'opolskie', 'podkarpackie', 'podlaskie', 'pomorskie', 'slaskie', 'swietokrzyskie', 'warminsko_mazurskie', 'wielkopolskie', 'zachodniopomorskie']


,year,pitotalnet_N_total,pitotalnet_Nw_total,pitotalnet_mean,pitotalnet_median,pitotalnet_p10,pitotalnet_p25,pitotalnet_p75,pitotalnet_p90,pitotalnet_p99,...,national_pitotalnet_pos_gini,national_pilab_pens_theil,national_pilab_pens_gini,national_hitotalnet_theil,national_hitotalnet_gini,national_hitotalnet_pos_theil,national_hitotalnet_pos_gini,national_hilab_pens_theil,national_hilab_pens_gini,region
0,1999,5890,2917139.0,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,...,NaN,NaN,NaN,0.1964,0.3377,0.1964,0.3300,0.2253,0.3935,dolnoslaskie
1,1999,4389,2068864.0,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,...,NaN,NaN,NaN,0.1964,0.3377,0.1964,0.3300,0.2253,0.3935,kujawsko_pomorskie
2,1999,6451,2637438.0,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,...,NaN,NaN,NaN,0.1964,0.3377,0.1964,0.3300,0.2253,0.3935,lodzkie
3,1999,4513,2209083.0,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,...,NaN,NaN,NaN,0.1964,0.3377,0.1964,0.3300,0.2253,0.3935,lubelskie
4,1999,2171,993422.0,0.0000,0.0,0.0,0.0,0.0000,0.0,0.0,...,NaN,NaN,NaN,0.1964,0.3377,0.1964,0.3300,0.2253,0.3935,lubuskie
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,2023,6025,4316591.0,39469.5627,38400.0,0.0,22800.0,52002.0000,72000.0,144000.0,...,0.294,0.1572,0.4643,0.2277,0.3666,0.2277,0.3577,0.2268,0.4070,slaskie
332,2023,2434,1156686.0,31304.2269,32400.0,0.0,14400.0,45360.0000,59040.0,116400.0,...,0.294,0.1572,0.4643,0.2277,0.3666,0.2277,0.3577,0.2268,0.4070,swietokrzyskie
333,2023,2357,1357910.0,31652.1144,32400.0,0.0,6000.0,44400.0000,60000.0,120000.0,...,0.294,0.1572,0.4643,0.2277,0.3666,0.2277,0.3577,0.2268,0.4070,warminsko_mazurskie
334,2023,4523,3478696.0,33367.1633,33600.0,0.0,7200.0,48000.0000,61200.0,120000.0,...,0.294,0.1572,0.4643,0.2277,0.3666,0.2277,0.3577,0.2268,0.4070,wielkopolskie


---
## Parse LA group emails → groups DataFrame

Scans all emails with **"LA group"** in the subject, extracts every `data_group_YEAR_Group_NNN = { … }` block, and assembles a tidy DataFrame.  One row = one (year, group).  The `region` column contains the full group label, e.g. `"Group 114"`.


In [8]:
import ast
import re
from datetime import datetime, timezone, timedelta

# ── Years of interest ────────────────────────────────────────────────────────
LA_GROUP_YEARS = {1999} | set(range(2004, 2024))   # same universe as voiv

# ── Regex: matches   data_group_1999_Group_114 = {  ...  \n}
# Group name is   Group_NNN   (capital G, underscore, digits)
_GROUP_BLOCK_RE = re.compile(
    r"data_group_(\d{4})_(Group_\d+)\s*=\s*\{(.+?)\n\}",
    re.DOTALL,
)


def _group_label(raw_name: str) -> str:
    """Convert 'Group_114' → 'Group 114'."""
    return raw_name.replace("_", " ", 1)   # replace only the first underscore


def extract_group_dicts(text: str) -> list[dict]:
    """Find all data_group_YEAR_Group_NNN blocks and return list of flat dicts."""
    rows = []
    for m in _GROUP_BLOCK_RE.finditer(text):
        year_str, raw_name, body = m.group(1), m.group(2), m.group(3)
        row = _parse_dict_body(body)      # reuse helper from cell 9
        row["year"]   = int(year_str)
        row["region"] = _group_label(raw_name)
        rows.append(row)
    return rows


# ── Scan emails ──────────────────────────────────────────────────────────────
all_group_rows = []
group_mail_index = []

for emlx_path in mail_root.rglob("*.emlx"):
    if emlx_path.name.endswith(".partial.emlx"):
        continue
    raw_bytes = emlx_path.read_bytes()
    if b"postbox@lisdatacenter.org" not in raw_bytes:
        continue

    msg = parse_emlx(emlx_path)
    if msg is None:
        continue

    subject = str(msg.get("Subject", ""))
    if "LA group" not in subject:
        continue

    year_match = re.search(r"\b(1999|200[4-9]|20[12][0-9])\b", subject)
    if year_match is None:
        continue
    year_in_subj = int(year_match.group(1))
    if year_in_subj not in LA_GROUP_YEARS:
        continue

    body = get_body(msg)
    rows = extract_group_dicts(body)
    if not rows:
        continue

    date_str = str(msg.get("Date", ""))
    try:
        mail_dt = parsedate_to_datetime(date_str)
        if mail_dt.tzinfo is None:
            mail_dt = mail_dt.replace(tzinfo=timezone.utc)
    except Exception:
        mail_dt = None

    for row in rows:
        row["_mail_subject"] = subject
        row["_mail_date"]    = mail_dt

    all_group_rows.extend(rows)
    group_mail_index.append({
        "subject": subject,
        "date":    mail_dt,
        "n_rows":  len(rows),
        "file":    str(emlx_path),
    })

print(f"Parsed {len(all_group_rows)} group-year rows from {len(group_mail_index)} emails.")
print("\nEmails used:")
for e in sorted(group_mail_index, key=lambda x: (x["date"] or datetime.min.replace(tzinfo=timezone.utc))):
    print(f"  [{e['date']}]  {e['subject']:<45}  → {e['n_rows']} rows")


Parsed 832 group-year rows from 21 emails.

Emails used:
  [2026-03-08 04:53:11+01:00]  job 1453440 LA group 2004                      → 40 rows
  [2026-03-08 04:55:01+01:00]  job 1453439 LA group 1999                      → 41 rows
  [2026-03-08 04:56:26+01:00]  job 1453441 LA group 2005                      → 40 rows
  [2026-03-08 05:04:40+01:00]  job 1453442 LA group 2006                      → 40 rows
  [2026-03-08 05:08:23+01:00]  job 1453443 LA group 2008                      → 40 rows
  [2026-03-08 05:13:52+01:00]  job 1453444 LA group 2009                      → 40 rows
  [2026-03-08 05:19:27+01:00]  job 1453445 LA group 2010                      → 40 rows
  [2026-03-08 05:24:54+01:00]  job 1453446 LA group 2011                      → 40 rows
  [2026-03-08 05:30:29+01:00]  job 1453447 LA group 2012                      → 40 rows
  [2026-03-08 05:35:57+01:00]  job 1453448 LA group 2013                      → 39 rows
  [2026-03-08 05:41:30+01:00]  job 1453449 LA group 2014       

In [9]:
df_group_raw = pd.DataFrame(all_group_rows)

if df_group_raw.empty:
    print("No data extracted.")
else:
    df_group_raw = df_group_raw.sort_values("_mail_date", na_position="first")
    df_groups = (
        df_group_raw
        .drop_duplicates(subset=["year", "region"], keep="last")
        .drop(columns=["_mail_subject", "_mail_date"])
        .sort_values(["year", "region"])
        .reset_index(drop=True)
    )

    skip_cols = {"year", "region"}
    for col in df_groups.columns:
        if col in skip_cols:
            continue
        df_groups[col] = pd.to_numeric(df_groups[col], errors="coerce")

    print(f"Final DataFrame: {df_groups.shape[0]} rows × {df_groups.shape[1]} columns")
    print(f"Years covered : {sorted(df_groups['year'].unique())}")
    print(f"Groups        : {sorted(df_groups['region'].unique())}")
    display(df_groups)

df_groups.to_csv(data_root.parent / "LIS_Groups_2000.csv", index=False)


Final DataFrame: 832 rows × 200 columns
Years covered : [np.int64(1999), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Groups        : ['Group 10', 'Group 100', 'Group 102', 'Group 104', 'Group 106', 'Group 108', 'Group 110', 'Group 112', 'Group 114', 'Group 115', 'Group 116', 'Group 117', 'Group 118', 'Group 127', 'Group 129', 'Group 13', 'Group 131', 'Group 133', 'Group 135', 'Group 137', 'Group 139', 'Group 141', 'Group 145', 'Group 146', 'Group 148', 'Group 15', 'Group 150', 'Group 154', 'Group 155', 'Group 159', 'Group 161', 'Group 163', 'Group 165', 'Group 167', 'Group 169', 'Group 171', 'Group 175', 'Group 176', 'Group 178', 'Group 18', 'Group 180', 'Group 182', 'Group 185', 'Group 186', 'Group 187', 'Group 19

,year,pitotalnet_N_total,pitotalnet_Nw_total,pitotalnet_mean,pitotalnet_median,pitotalnet_p10,pitotalnet_p25,pitotalnet_p75,pitotalnet_p90,pitotalnet_p99,...,national_pitotalnet_pos_gini,national_pilab_pens_theil,national_pilab_pens_gini,national_hitotalnet_theil,national_hitotalnet_gini,national_hitotalnet_pos_theil,national_hitotalnet_pos_gini,national_hilab_pens_theil,national_hilab_pens_gini,region
0,1999,3479,1.371836e+06,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,0.2037,0.3413,0.2037,0.3337,0.2341,0.3938,Group 114
1,1999,5168,3.075978e+06,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,0.2037,0.3413,0.2037,0.3337,0.2341,0.3938,Group 115
2,1999,304,1.143165e+05,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,0.2037,0.3413,0.2037,0.3337,0.2341,0.3938,Group 116
3,1999,525,1.966741e+05,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,0.2037,0.3413,0.2037,0.3337,0.2341,0.3938,Group 117
4,1999,1811,9.432360e+05,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,0.2037,0.3413,0.2037,0.3337,0.2341,0.3938,Group 145
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
827,2023,1398,7.208440e+05,36144.8419,35400.0,0.0,18960.0,49200.0,69600.0,120000.0,...,0.2897,0.1518,0.4614,0.2288,0.3673,0.2288,0.3588,0.2251,0.4065,Group 72
828,2023,1322,6.053495e+05,39335.7480,38400.0,0.0,20088.0,54000.0,72000.0,144000.0,...,0.2897,0.1518,0.4614,0.2288,0.3673,0.2288,0.3588,0.2251,0.4065,Group 76
829,2023,2420,1.710504e+06,30188.2198,31920.0,0.0,0.0,43200.0,56400.0,120000.0,...,0.2897,0.1518,0.4614,0.2288,0.3673,0.2288,0.3588,0.2251,0.4065,Group 82
830,2023,1215,7.140471e+05,52400.6504,48000.0,0.0,26220.0,66960.0,97200.0,235200.0,...,0.2897,0.1518,0.4614,0.2288,0.3673,0.2288,0.3588,0.2251,0.4065,Group 85
